# UC Table & Column Metadata Diagnosis

### Source : KNOWLEDGE_RETRIEVAL_3/uc_metadata_diagnosis.ipynb

### Maps to exam bullets:
- Given an underperforming agent and its UC metadata, identify which missing/poorly-specified
  metadata elements cause the accuracy gap, and pick the highest-impact config change.
- Given documented retrieval problems, diagnose the failure mode from MLflow eval logs + UC metadata,
  and pick the UC governance action that most directly resolves it.



### The idea

A text-to-SQL agent can only be as precise as the **encodings** in your data are documented.
Business databases are full of integer/code columns (`tier_code`, `region_code`, status flags)
whose meaning lives only in someone's head — or in a UC column `COMMENT`. 


## Setup
A `.env` file with `DATABRICKS_HOST` and `DATABRICKS_TOKEN`

In [ ]:
import os
from dotenv import load_dotenv
import sys
sys.path.append(os.path.abspath(".."))
from _utils.sql_utils import sql_call

load_dotenv()

assert os.getenv("DATABRICKS_HOST"), "DATABRICKS_HOST not set -- check your .env file"
assert os.getenv("DATABRICKS_TOKEN"), "DATABRICKS_TOKEN not set -- check your .env file"
assert os.getenv("SQL_WAREHOUSE_ID"), "SQL_WAREHOUSE_ID not set -- check your .env file"
assert os.getenv("MLFLOW_TRACKING_URI"), "MLFLOW_TRACKING_URI not set -- check your .env file"

## Imports and config

In [ ]:
import json
import mlflow
import pandas as pd
from databricks.sdk import WorkspaceClient
from databricks_langchain import ChatDatabricks
from langchain_core.messages import HumanMessage, SystemMessage

CATALOG, SCHEMA = "demo", "context"
WAREHOUSE_ID = os.getenv("SQL_WAREHOUSE_ID")
TABLE = f"{CATALOG}.{SCHEMA}.metadata_demo_customers"

w   = WorkspaceClient()
llm = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct")

# TODO: change "oliver@mlops-media.com" to your own workspace user email
mlflow.set_experiment("/Users/oliver@mlops-media.com/uc_metadata_diagnosis")
print(f"Workspace host: {w.config.host}")

---
## 1 - A small scratch table with two *encoded* columns

`tier_code` and `region_code` store integers/short codes instead of human-readable values --
extremely common in real warehouses. We create this as a standalone scratch table so we never
touch the real `customers` / `shipping_zones` tables curated into the Genie space.


In [ ]:
ddl_create = f"""
CREATE OR REPLACE TABLE {TABLE} (
    customer_id STRING,
    last_name   STRING,
    tier_code   INT,
    region_code STRING
)
"""
sql_call(databricks_sdk_client=w, statement=ddl_create, warehouse_id=WAREHOUSE_ID)

ddl_insert = f"""
INSERT INTO {TABLE} VALUES
    ('C-1001', 'Smith',   1, 'A'),
    ('C-1002', 'Jones', 2, 'A'),
    ('C-1013', 'Smith',   3, 'B'),
    ('C-1008', 'Lee',     2, 'C'),
    ('C-1005', 'Davis',   3, 'A'),
    ('C-1006', 'Garcia',  1, 'B')
"""
sql_call(databricks_sdk_client=w, statement=ddl_insert, warehouse_id=WAREHOUSE_ID)
print(f"Created and populated {TABLE}")
print("Encoding (known only to us right now): tier_code 1=standard, 2=premium, 3=enterprise")
print("                                       region_code A=domestic, B=nearby intl, C=remote intl")

## 2 - Eval set with ground truth

Each question has a known correct set of `customer_id`s, computed directly from the encoding
above -- not from whatever the LLM thinks the encoding is.


In [ ]:
eval_set = [
    {"question": "Which customers are on the premium tier?",                 "expected_ids": {"C-1002", "C-1008"}},
    {"question": "List customers on the enterprise tier.",                   "expected_ids": {"C-1013", "C-1005"}},
    {"question": "Show standard-tier customers in the domestic region.",     "expected_ids": {"C-1001"}},
]
print(f"Eval set size: {len(eval_set)}")

## 3 - Build a schema context from `information_schema`, generate SQL, execute it

We read the *actual current* column comments straight from Unity Catalog -- the same source
Genie itself reads -- so the only thing that changes between the two eval runs is what's stored
in UC, not anything in our prompt.


In [ ]:
DATABRICKS_CLIENT = w
def get_schema_context(table, catalog, schema):
    table_name = table.split(".")[-1]
    rows = sql_call(
        databricks_sdk_client=DATABRICKS_CLIENT, 
        statement=f"""
        SELECT column_name, data_type, comment
        FROM {catalog}.information_schema.columns
        WHERE table_catalog = '{catalog}' AND table_schema = '{schema}' AND table_name = '{table_name}'
        ORDER BY ordinal_position
        """,
        warehouse_id=WAREHOUSE_ID,
    )
    lines = []
    for r in rows:
        comment = r.get("comment") or "(no comment)"
        lines.append(f"- {r['column_name']} ({r['data_type']}): {comment}")
    return f"Table {table}\nColumns:\n" + "\n".join(lines)

In [ ]:
def generate_sql(question, schema_context):
    system = (
        "You write Databricks SQL. Use ONLY the table and columns described below. "
        "Respond with a single SELECT statement only -- no markdown, no explanation.\n\n"
        f"{schema_context}"
    )
    response = llm.invoke([SystemMessage(content=system), HumanMessage(content=question)])
    sql_text = response.content.strip().strip("`")
    if sql_text.lower().startswith("sql"):
        sql_text = sql_text[3:].strip()
    return sql_text

In [ ]:
def run_generated_sql(sql_text):
    try:
        rows = sql_call(        
            databricks_sdk_client=DATABRICKS_CLIENT, 
            statement=sql_text, 
            warehouse_id=WAREHOUSE_ID
        )
        return {row["customer_id"] for row in rows if "customer_id" in row}
    except Exception as e:
        return {"__error__": str(e)}

In [ ]:
def run_eval(eval_set, schema_context, run_name):
    rows = []
    with mlflow.start_run(run_name=run_name):
        for item in eval_set:
            sql_text = generate_sql(item["question"], schema_context)
            returned_ids = run_generated_sql(sql_text)
            correct = returned_ids == item["expected_ids"]
            rows.append({
                "question": item["question"], "generated_sql": sql_text,
                "expected_ids": sorted(item["expected_ids"]), "returned_ids": sorted(returned_ids),
                "correct": correct,
            })

        accuracy = sum(r["correct"] for r in rows) / len(rows)
        mlflow.log_metric("sql_correctness", accuracy)
        mlflow.log_table(data=pd.DataFrame(rows), artifact_file="eval_rows.json")

    return rows, accuracy

## 4 - Run the eval with NO column comments (current state)


In [ ]:
poor_schema_context = get_schema_context(TABLE, CATALOG, SCHEMA)
print(poor_schema_context)
print()

poor_rows, poor_accuracy = run_eval(eval_set, poor_schema_context, run_name="no_column_comments")

print(f"=== NO column comments -- SQL correctness: {poor_accuracy:.0%} ===")
for r in poor_rows:
    status = "OK" if r["correct"] else "MISS"
    print(f"  [{status}] {r['question']}")
    print(f"        SQL: {r['generated_sql']}")
    print(f"        expected={r['expected_ids']}  got={r['returned_ids']}")

## 5 - Add column comments documenting the encodings, then re-run

Same table, same rows -- only `COMMENT` changes. This is exactly the "AI Generate column
descriptions" action from the Genie walkthrough in notebook 2, applied here in SQL.


In [ ]:
sql_call(
        databricks_sdk_client=DATABRICKS_CLIENT, 
        statement=f"ALTER TABLE {TABLE} ALTER COLUMN tier_code COMMENT 'Membership tier as an integer code: 1 = standard, 2 = premium, 3 = enterprise.'",
        warehouse_id=WAREHOUSE_ID
    )

sql_call(
        databricks_sdk_client=DATABRICKS_CLIENT, 
        statement=f"ALTER TABLE {TABLE} ALTER COLUMN region_code COMMENT "
        f"'Shipping region as a single-letter code: A = domestic, B = nearby international, "
        f"C = remote international.'",
        warehouse_id=WAREHOUSE_ID
        )

print(f"Added column comments to {TABLE}")

In [ ]:
rich_schema_context = get_schema_context(TABLE, CATALOG, SCHEMA)
print(rich_schema_context)
print()

rich_rows, rich_accuracy = run_eval(eval_set, rich_schema_context, run_name="with_column_comments")

print(f"=== WITH column comments -- SQL correctness: {rich_accuracy:.0%} ===")
for r in rich_rows:
    status = "OK" if r["correct"] else "MISS"
    print(f"  [{status}] {r['question']}")
    print(f"        SQL: {r['generated_sql']}")
    print(f"        expected={r['expected_ids']}  got={r['returned_ids']}")

## 6 - Diagnose the failure mode and pick the highest-impact fix

Bullet 8: don't just report a score, use the MLflow-logged rows to say *why* it failed. Bullet 1:
map that to a single, highest-impact UC config change.


In [ ]:
print("=== Diagnosis ===")
print(f"  SQL correctness: {poor_accuracy:.0%} -> {rich_accuracy:.0%}")
print()
print("tier_code and region_code are encoded columns with NO UC comment.")
print("The LLM has no way to know that 2 means 'premium' or that 'B' means.")
print("=== Solution ===")

print("Add comments!")